# CONUS Visualizations — SIF × Irrigation × Drought

This notebook assembles all primary CONUS-scale visualizations for the
SIF × HumanET × Drought analysis. It loads pre-computed results from:

- **`01_human_et_conus.ipynb`** → processed OpenET and NLDAS ET files used to
  reconstruct `delta_maps` (HumanET = OpenET − NLDAS Noah ET)
- **`02_irrigation_sif_regression_conus.ipynb`** → `df_combined_gs.parquet`,
  `crop_mask_static.npy`, and pixel-level regression slope results
  (recomputed here if not saved to disk)

**Prerequisites:** Run notebooks `01_` and `02_` first to generate the
parquet and mask files in `data/processed/conus/regression/`.

In [1]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from scipy import stats
import rasterio
import xarray as xr

_root_env    = os.environ.get('SIF_ROOT')
project_root = Path(_root_env) if _root_env else Path('../../..').resolve()

proc = project_root / 'data' / 'processed' / 'conus'
figs = project_root / 'figures' / 'conus'
figs.mkdir(parents=True, exist_ok=True)

openet_proc = proc / 'openet'
nldas_proc  = proc / 'nldas'

print('Project root:', project_root)
print('Data dir:    ', proc)
print('Figures dir: ', figs)

Project root: /home/pielab-sandbox-jcoldiron/SIF-Analysis
Data dir:     /home/pielab-sandbox-jcoldiron/SIF-Analysis/data/processed/conus
Figures dir:  /home/pielab-sandbox-jcoldiron/SIF-Analysis/figures/conus


## 1. Setup and Data Loading

We reconstruct the key data objects needed for visualization:

1. **Config constants** — identical to those in `01_` and `02_` so all grids align
2. **`df_combined_gs`** — the pixel-month panel dataset (parquet)
3. **`crop_mask_static`** — static 189 × 325 boolean cropland mask (numpy)
4. **`delta_maps`** — reconstructed from the processed OpenET and NLDAS ET files
5. **Pixel regression slopes** — loaded from cache or recomputed from `df_combined_gs`

### 1.1 Configuration

In [2]:
# ── Grid and analysis constants (must match 01_ and 02_) ──────────────────
CONUS_LON = np.arange(-124.6875, -84.0625,  0.125)   # 325 longitudes
CONUS_LAT = np.arange(  49.3125,  25.6875, -0.125)   # 189 latitudes (N→S)
n_lat, n_lon = len(CONUS_LAT), len(CONUS_LON)

YEARS          = list(range(2015, 2025))   # 10-year study period
GROWING_SEASON = [4, 5, 6, 7, 8, 9]       # April–September

IRR_THRESHOLD_MM   = 20.0   # mm/month — HumanET > 20 = irrigation
DROUGHT_SPEI_THRESH = -0.5  # SPEI ≤ -0.5 = at least mild drought
MIN_OBS_PIXEL       = 5     # minimum drought-month obs per pixel for regression
ALPHA_SPATIAL       = 0.10  # significance threshold for pixel slopes

print('CONUS grid:', n_lon, 'x', n_lat, 'at 0.125 deg')
print('Study period:', YEARS[0], '-', YEARS[-1])
print('Growing season months:', GROWING_SEASON)

CONUS grid: 325 x 189 at 0.125 deg
Study period: 2015 - 2024
Growing season months: [4, 5, 6, 7, 8, 9]


### 1.2 Load Panel Dataset and Cropland Mask

In [3]:
parquet_path = proc / 'regression' / 'df_combined_gs.parquet'
mask_path    = proc / 'regression' / 'crop_mask_static.npy'

if not parquet_path.exists():
    raise FileNotFoundError(
        'Run 01_human_et_conus.ipynb first: ' + str(parquet_path)
    )

df = pd.read_parquet(parquet_path)
if 'date' not in df.columns:
    df['date'] = pd.to_datetime(df['yyyymm'], format='%Y%m')
if 'dm_cat_int' not in df.columns and 'dm_cat' in df.columns:
    df['dm_cat_int'] = df['dm_cat'].round().astype('Int64')

crop_mask_static = np.load(mask_path)

print('Loaded:', parquet_path.name)
print('Shape:', df.shape)
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
print('Cropland pixels (mask):', crop_mask_static.sum())

Loaded: df_combined_gs.parquet
Shape: (1509180, 12)
Date range: 2015-04-01 to 2024-09-01
Cropland pixels (mask): 25153


### 1.3 Reconstruct delta_maps (HumanET = OpenET − NLDAS ET)

We reload the processed OpenET and NLDAS ET files for **growing-season months only**
(April–September, 2015–2024) and recompute HumanET = OpenET − NLDAS ET. Files are
already aligned to the 189 × 325 CONUS grid, so no reprojection is needed.
We use `crop_mask_static` for masking (same ≥ 50% cropland threshold as notebook 01).

In [4]:
delta_maps = {}   # key: (year, month) → 2D array (189, 325)
n_proc = 0
n_miss = 0

for year in YEARS:
    for month in GROWING_SEASON:
        yyyymm = str(year) + str(month).zfill(2)
        fp_openet = openet_proc / ('OpenET_CONUS_' + yyyymm + '.tif')
        fp_nldas  = nldas_proc  / ('NLDAS_Evap_'   + yyyymm + '.nc')

        openet_arr = None
        nldas_arr  = None

        if fp_openet.exists():
            try:
                with rasterio.open(fp_openet) as src:
                    a  = src.read(1).astype(float)
                    nd = src.nodata
                    if nd is not None:
                        a[a == nd] = np.nan
                    if a.shape == (n_lat, n_lon):
                        openet_arr = a
            except Exception as e:
                print('  OpenET error ' + yyyymm + ': ' + str(e))

        if fp_nldas.exists():
            try:
                ds = xr.open_dataset(fp_nldas)
                et_var = None
                for vname in ['Evap', 'EVP', 'et', 'ET']:
                    if vname in ds:
                        et_var = vname
                        break
                if et_var is None:
                    et_var = list(ds.data_vars)[0]
                arr_n = ds[et_var].values
                ds.close()
                if arr_n.ndim == 3:
                    arr_n = arr_n[0]
                if arr_n.shape == (n_lat, n_lon):
                    nldas_arr = arr_n.astype(float)
            except Exception as e:
                print('  NLDAS error ' + yyyymm + ': ' + str(e))

        if openet_arr is not None and nldas_arr is not None:
            delta = openet_arr - nldas_arr
            delta = np.where(crop_mask_static, delta, np.nan)
            delta = np.where(np.abs(delta) > 500, np.nan, delta)
            delta_maps[(year, month)] = delta
            n_proc += 1
        else:
            n_miss += 1

print('Growing-season months loaded: ' + str(n_proc))
print('Missing months:               ' + str(n_miss))
print('delta_maps entries:           ' + str(len(delta_maps)))

Growing-season months loaded: 60
Missing months:               0
delta_maps entries:           60


### 1.4 Pixel-Level Regression Slopes

We check for a cached results file (`pix_slope_results.csv`) written by notebook 02.
If not found we recompute using the identical logic: OLS of `sif_z ~ delta_et`
restricted to drought months (SPEI ≤ −0.5), one regression per cropland pixel,
minimum 5 observations, retaining results at p < 0.10.

In [5]:
_slope_cache = proc / 'regression' / 'pix_slope_results.csv'

if _slope_cache.exists():
    pix_stats = pd.read_csv(_slope_cache)
    print('Loaded pixel slopes from cache:', _slope_cache.name)
else:
    print('Cache not found — recomputing pixel-level slopes...')

    # Build complete-case regression dataset (same as notebook 02)
    df_reg = df.dropna(subset=['sif_z', 'spei90d', 'delta_et']).copy()
    df_reg = df_reg[
        np.isfinite(df_reg['sif_z']) &
        np.isfinite(df_reg['spei90d']) &
        np.isfinite(df_reg['delta_et'])
    ].copy()

    # Filter to drought months only
    df_drought = df_reg[df_reg['spei90d'] <= DROUGHT_SPEI_THRESH].copy()
    print('Drought-month obs:', len(df_drought))

    def _pixel_slope(grp):
        xy = grp[['delta_et', 'sif_z']].dropna()
        if len(xy) < MIN_OBS_PIXEL:
            return pd.Series({'slope': np.nan, 'pval': np.nan, 'n': len(xy)})
        slope, _, _, pval, _ = stats.linregress(
            xy['delta_et'].values, xy['sif_z'].values
        )
        return pd.Series({'slope': slope, 'pval': pval, 'n': len(xy)})

    print('Running pixel regressions (~30-60 s)...')
    pix_stats = (
        df_drought
        .groupby(['lat', 'lon'])
        .apply(_pixel_slope)
        .reset_index()
    )
    print('Pixel regression complete.')

# Significant pixels at p < ALPHA_SPATIAL
pix_sig = pix_stats[pix_stats['pval'] < ALPHA_SPATIAL].copy()
n_pos   = int((pix_sig['slope'] > 0).sum())
n_neg   = int((pix_sig['slope'] < 0).sum())

print()
print('Pixels with >= ' + str(MIN_OBS_PIXEL) + ' drought obs: '
      + str(pix_stats['slope'].notna().sum()))
print('Significant at p < ' + str(ALPHA_SPATIAL) + ': ' + str(len(pix_sig)))
print('  Positive slope (buffering): ' + str(n_pos))
print('  Negative slope:             ' + str(n_neg))

Cache not found — recomputing pixel-level slopes...
Drought-month obs: 120766
Running pixel regressions (~30-60 s)...


Pixel regression complete.

Pixels with >= 5 drought obs: 7456
Significant at p < 0.1: 2032
  Positive slope (buffering): 1701
  Negative slope:             331


---

## 2. Human ET Change Over Time Map

This map shows **where HumanET (OpenET − NLDAS ET) has been increasing or decreasing
over the 2015–2024 study period** across CONUS cropland.

**Method:**
1. For each year, compute the mean of the six growing-season monthly HumanET values
   at each cropland pixel, giving a single annual estimate (mm/month).
2. Fit a linear trend (OLS) across the 10 annual means using `scipy.stats.linregress`.
3. Report the slope in **mm/month per year** — positive = HumanET increasing,
   negative = decreasing.

**Interpretation:** A positive trend may reflect expanded irrigation, improved
crop water use, or wetter conditions driving higher OpenET. A negative trend can
indicate crop abandonment, drought-limited production, or switching to less
water-intensive crops.

### 2.1 Compute Per-Pixel Trend Slopes

In [6]:
# ── Per-year mean growing-season HumanET at each pixel ───────────────────
yearly_gs = {}
for year in YEARS:
    arrs = [delta_maps[(year, m)] for m in GROWING_SEASON
            if (year, m) in delta_maps]
    if arrs:
        yearly_gs[year] = np.nanmean(np.stack(arrs, axis=0), axis=0)
    else:
        yearly_gs[year] = np.full((n_lat, n_lon), np.nan)

# Stack: shape (n_years, n_lat, n_lon)
gs_stack  = np.stack([yearly_gs[y] for y in YEARS], axis=0)
year_vals = np.array(YEARS, dtype=float)
print('Yearly GS mean stack shape:', gs_stack.shape)

# ── Linear trend at each cropland pixel ───────────────────────────────────
trend_slope = np.full((n_lat, n_lon), np.nan)
trend_pval  = np.full((n_lat, n_lon), np.nan)

crop_rows, crop_cols = np.where(crop_mask_static)
for ri, ci in zip(crop_rows, crop_cols):
    y     = gs_stack[:, ri, ci]
    valid = np.isfinite(y)
    if valid.sum() < 3:
        continue
    slope, _, _, pval, _ = stats.linregress(year_vals[valid], y[valid])
    trend_slope[ri, ci] = slope
    trend_pval[ri, ci]  = pval

# ── Summary statistics ─────────────────────────────────────────────────────
finite_slopes = trend_slope[np.isfinite(trend_slope)]
mean_trend    = float(np.nanmean(finite_slopes))
frac_inc      = float((finite_slopes > 0).mean())
frac_dec      = 1.0 - frac_inc

print('Trend computed for ' + str(len(finite_slopes)) + ' cropland pixels')
print('Slope range: ' + str(round(float(finite_slopes.min()), 3))
      + ' to ' + str(round(float(finite_slopes.max()), 3)) + ' mm/month/year')
print('CONUS mean trend:  ' + str(round(mean_trend, 4)) + ' mm/month/year')
print('Pixels increasing: ' + str(round(100 * frac_inc, 1)) + '%')
print('Pixels decreasing: ' + str(round(100 * frac_dec, 1)) + '%')

Yearly GS mean stack shape: (10, 189, 325)


/tmp/ipykernel_2975427/1085650207.py:7: RuntimeWarning: Mean of empty slice
  yearly_gs[year] = np.nanmean(np.stack(arrs, axis=0), axis=0)


Trend computed for 7595 cropland pixels
Slope range: -20.574 to 11.72 mm/month/year
CONUS mean trend:  0.8313 mm/month/year
Pixels increasing: 80.0%
Pixels decreasing: 20.0%


### 2.2 CONUS Trend Map

In [7]:
# ── NOTE: this standalone trend map is kept for quick reference.
# The publication-quality two-panel figure is in Section 4 below.
# Requires Section 4.1 helper cells (_X5070, _Y5070, _draw_states,
# _add_basemap, _albers_axes). Run Section 4.1 first if standalone.
# ─────────────────────────────────────────────────────────────────────────

try:
    _X5070
except NameError:
    from pyproj import Transformer as _Tr
    _t5070 = _Tr.from_crs('EPSG:4326', 'EPSG:5070', always_xy=True)
    _LON_GRID, _LAT_GRID = np.meshgrid(CONUS_LON, CONUS_LAT)
    _X5070, _Y5070 = _t5070.transform(_LON_GRID, _LAT_GRID)
    _xmin, _xmax = float(_X5070.min()) - 80_000, float(_X5070.max()) + 80_000
    _ymin, _ymax = float(_Y5070.min()) - 80_000, float(_Y5070.max()) + 80_000

# ── Color scale ───────────────────────────────────────────────────────────
_fs = trend_slope[np.isfinite(trend_slope)]
_vmax = max(float(np.nanpercentile(np.abs(_fs), 97)), 0.10)

_trend_masked = np.where(crop_mask_static, trend_slope, np.nan)

_cmap_t = plt.cm.RdBu.copy()
_cmap_t.set_bad('white', alpha=0)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4.5))

im = ax.pcolormesh(
    _X5070, _Y5070, _trend_masked,
    cmap=_cmap_t, vmin=-_vmax, vmax=_vmax,
    shading='nearest', rasterized=True, zorder=2,
)

# Apply axes formatting (inline fallback if Section 4.1 not yet run)
try:
    _albers_axes(ax)
except NameError:
    ax.set_xticks([]); ax.set_yticks([])
    for _sp in ax.spines.values(): _sp.set_visible(False)
    ax.set_aspect('equal')
    ax.set_xlim(_xmin, _xmax)
    ax.set_ylim(_ymin, _ymax)

try:
    _add_basemap(ax)
except NameError:
    pass

try:
    _draw_states(ax, lw=0.4, color='#333333')
except NameError:
    pass

cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, shrink=0.85)
cb.set_label('HumanET linear trend (mm/month/year)  |  Blue = increasing  /  Red = decreasing',
             fontsize=9)

_title_str = (
    'Linear Trend in Growing-Season HumanET (OpenET - NLDAS ET) 2015-2024 - CONUS Cropland'
)
_sub_str = (
    'CONUS mean: {:.3f} mm/month/yr  |  Increasing: {:.1f}%  |  Decreasing: {:.1f}%'.format(
        mean_trend, 100 * frac_inc, 100 * frac_dec
    )
)
ax.set_title(_title_str + '\n' + _sub_str, fontsize=10)

plt.tight_layout()
plt.savefig(figs / 'human_et_trend_map.png', dpi=300, bbox_inches='tight')
plt.savefig(figs / 'human_et_trend_map.pdf', bbox_inches='tight')
plt.show()
print('Saved: human_et_trend_map.png / .pdf  (Albers EPSG:5070, 300 DPI)')


Saved: human_et_trend_map.png / .pdf  (Albers EPSG:5070, 300 DPI)


---

## 3. Positive vs. Negative Pixel Regression Slopes

This map shows **where irrigation is statistically associated with higher SIF
z-scores during drought months** across CONUS cropland.

Each pixel represents an independent OLS regression of SIF anomaly (`sif_z`) on
HumanET (`delta_et`), estimated using drought-month observations only
(SPEI-90d ≤ −0.5, ≥ 5 observations per pixel). Significance threshold: p < 0.10.

**Color encoding:**
- **Blue** — positive slope: more HumanET → higher SIF z-score during drought
  (irrigation buffers crop photosynthesis under water stress)
- **Red** — negative slope: more HumanET → lower SIF z-score
  (no buffering or counter-intuitive signal, e.g. stressed irrigated fields
  in areas with severe groundwater depletion)
- **Light gray** — non-significant cropland pixels (p ≥ 0.10 or < 5 drought obs)

This is a descriptive spatial analysis; causal interpretation requires the
pooled regression in notebook 02.

In [8]:
# ── Build lat/lon → row/col lookup for fast grid placement ───────────────
_lat_to_row = {round(float(v), 4): i for i, v in enumerate(CONUS_LAT)}
_lon_to_col = {round(float(v), 4): i for i, v in enumerate(CONUS_LON)}

# ── Categorical map: NaN=outside cropland, 0=non-sig, +1=pos, -1=neg ─────
cat_map = np.full((n_lat, n_lon), np.nan)
cat_map[crop_mask_static] = 0.0      # all cropland pixels start as non-sig

n_placed = 0
for _, row in pix_sig.iterrows():
    ri = _lat_to_row.get(round(float(row['lat']), 4))
    ci = _lon_to_col.get(round(float(row['lon']), 4))
    if ri is not None and ci is not None:
        cat_map[ri, ci] = 1.0 if row['slope'] > 0 else -1.0
        n_placed += 1

print('Significant pixels placed on grid: ' + str(n_placed) + ' / ' + str(len(pix_sig)))
print('  Positive (blue): ' + str(n_pos))
print('  Negative (red):  ' + str(n_neg))
print('  Non-sig (gray):  '
      + str(int((cat_map == 0).sum())))

# ── Custom 3-class colormap ────────────────────────────────────────────────
# Values: -1 → red, 0 → light gray, +1 → blue
_cmap   = ListedColormap(['#C62828', '#CCCCCC', '#1565C0'])  # red, gray, blue
_bounds = [-1.5, -0.5, 0.5, 1.5]
_norm   = BoundaryNorm(_bounds, _cmap.N)

fig, ax = plt.subplots(figsize=(17, 7))

ax.imshow(
    cat_map,
    extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
    origin='upper',
    cmap=_cmap,
    norm=_norm,
    aspect='auto',
)

# ── Legend ─────────────────────────────────────────────────────────────────
legend_elements = [
    Patch(facecolor='#1565C0', edgecolor='none',
          label='Positive slope — irrigation buffers SIF (n=' + str(n_pos) + ')'),
    Patch(facecolor='#C62828', edgecolor='none',
          label='Negative slope — no buffering benefit (n=' + str(n_neg) + ')'),
    Patch(facecolor='#CCCCCC', edgecolor='none',
          label='Non-significant cropland (p >= ' + str(ALPHA_SPATIAL)
                + ' or < ' + str(MIN_OBS_PIXEL) + ' obs)'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=10, framealpha=0.9)

ax.set_title(
    'Pixel-Level Irrigation Buffering of SIF Under Drought - CONUS Cropland\n'
    'OLS: SIF z-score ~ HumanET | Drought months (SPEI <= '
    + str(DROUGHT_SPEI_THRESH)
    + ')  |  Significant at p < '
    + str(ALPHA_SPATIAL),
    fontsize=12
)
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude',  fontsize=11)
ax.grid(alpha=0.15)

plt.tight_layout()
plt.savefig(figs / 'reg_sif_slope_positive_negative.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reg_sif_slope_positive_negative.png')

Significant pixels placed on grid: 2032 / 2032
  Positive (blue): 1701
  Negative (red):  331
  Non-sig (gray):  23121


Saved: reg_sif_slope_positive_negative.png


---

## 3.5 Diagnosis — Border Artifacts, Aspect Ratio, and State Boundaries

Before producing the publication figure we diagnose the three known problems in the
earlier draft maps:

1. **Border artifacts** — spurious NLDAS pixels appearing outside the US political boundary
   (especially southern AZ/NM/TX and northern WA/ID). A pixel can pass the 50% cropland CDL
   threshold even if only part of it falls inside the US, because CDL is masked by land cover,
   not by political boundary.

2. **Squished aspect ratio** — `imshow` with `aspect='auto'` in geographic coordinates
   (EPSG:4326) renders 1° lon ≡ 1° lat, which is incorrect at mid-latitudes.

3. **Missing state boundaries** — no political reference overlay.

### Diagnostic cells:
- **(a)** CDL cropland mask extent vs. US states dissolved boundary
- **(b)** CRS / transform / shape of each input dataset
- **(c)** Spot-check values in border-artifact regions
- **(d)** Diagnosis summary

In [9]:
# ── Diagnostic (a): CDL mask extent vs. US states dissolved boundary ─────
import geopandas as gpd

_states_fp = (project_root / 'data' / 'processed' / 'conus' / 'us_states_simple.geojson').resolve()

if _states_fp.exists():
    _states_gdf  = gpd.read_file(_states_fp)
    _us_dissolved = _states_gdf.dissolve()     # single polygon = full CONUS

    # Bounding box of US dissolved polygon
    _us_bounds = _us_dissolved.total_bounds   # (minx, miny, maxx, maxy)
    print('US boundary extent (EPSG:4326):')
    print('  lon: {:.4f} to {:.4f}'.format(_us_bounds[0], _us_bounds[2]))
    print('  lat: {:.4f} to {:.4f}'.format(_us_bounds[1], _us_bounds[3]))
else:
    print('WARNING: us_states_simple.geojson not found — skipping boundary diagnostics')
    _us_dissolved = None

# Bounding box of CONUS grid (all pixels)
print('\nCONUS NLDAS grid extent (EPSG:4326):')
print('  lon: {:.4f} to {:.4f}'.format(float(CONUS_LON[0] - 0.0625), float(CONUS_LON[-1] + 0.0625)))
print('  lat: {:.4f} to {:.4f}'.format(float(CONUS_LAT[-1] - 0.0625), float(CONUS_LAT[0]  + 0.0625)))

# Cropland mask pixels
_cmask_rows, _cmask_cols = np.where(crop_mask_static)
_cmask_lats = CONUS_LAT[_cmask_rows]
_cmask_lons = CONUS_LON[_cmask_cols]
print('\nCropland mask pixel extent (EPSG:4326):')
print('  lon: {:.4f} to {:.4f}'.format(float(_cmask_lons.min()), float(_cmask_lons.max())))
print('  lat: {:.4f} to {:.4f}'.format(float(_cmask_lats.min()), float(_cmask_lats.max())))

# Check for cropland pixels outside the US boundary
if _us_dissolved is not None:
    from shapely.geometry import box as _box, MultiPoint as _MP
    # Sample every 5th cropland pixel to keep it fast
    _sample_pts = gpd.GeoDataFrame(
        {'geometry': [_box(lo - 0.0625, la - 0.0625, lo + 0.0625, la + 0.0625)
                      for la, lo in zip(_cmask_lats[::5], _cmask_lons[::5])]},
        crs='EPSG:4326',
    )
    _inside = _sample_pts.within(_us_dissolved.geometry.iloc[0])
    n_out = int((~_inside).sum())
    n_tot = int(len(_inside))
    print('\n(a) {}/{} sampled CDL cropland pixels (every 5th) fall OUTSIDE US boundary'.format(
        n_out, n_tot
    ))
    if n_out > 0:
        print('    => Border artifacts confirmed: CDL mask alone is insufficient.')
    else:
        print('    => No border artifacts detected in sample.')

US boundary extent (EPSG:4326):
  lon: -124.7000 to -67.0000
  lat: 24.5000 to 49.0000

CONUS NLDAS grid extent (EPSG:4326):
  lon: -124.7500 to -84.1250
  lat: 25.7500 to 49.3750

Cropland mask pixel extent (EPSG:4326):
  lon: -124.6875 to -84.1875
  lat: 25.8125 to 49.3125

(a) 3391/5031 sampled CDL cropland pixels (every 5th) fall OUTSIDE US boundary
    => Border artifacts confirmed: CDL mask alone is insufficient.


In [10]:
# ── Diagnostic (b): CRS, transform, and shape of each input dataset ──────
print('(b) Dataset grid/CRS summary')
print()

# CONUS NLDAS grid (inferred from array constants)
print('NLDAS CONUS analysis grid:')
print('  Shape : {} lat x {} lon'.format(n_lat, n_lon))
print('  Res   : 0.125 deg')
print('  CRS   : EPSG:4326 (geographic, WGS84)')
print('  Extent: lon {:.4f}–{:.4f}, lat {:.4f}–{:.4f}'.format(
    float(CONUS_LON[0]), float(CONUS_LON[-1]),
    float(CONUS_LAT[-1]), float(CONUS_LAT[0])
))
print()

# Spot-check one OpenET GeoTIFF
_oe_sample = next(iter(sorted((openet_proc).glob('OpenET_CONUS_*.tif'))), None)
if _oe_sample:
    with rasterio.open(_oe_sample) as _src:
        print('OpenET GeoTIFF ({}):'.format(_oe_sample.name))
        print('  Shape    :', _src.height, 'x', _src.width)
        print('  CRS      :', _src.crs)
        print('  Transform:', _src.transform)
        print('  NoData   :', _src.nodata)
        print()

# Spot-check one NLDAS Noah NetCDF
_nldas_sample = next(iter(sorted((nldas_proc).glob('NLDAS_Evap_*.nc'))), None)
if _nldas_sample:
    _ds = xr.open_dataset(_nldas_sample)
    _lat_arr = _ds.coords.get('lat', _ds.coords.get('latitude', None))
    _lon_arr = _ds.coords.get('lon', _ds.coords.get('longitude', None))
    print('NLDAS Noah NetCDF ({}):'.format(_nldas_sample.name))
    print('  Variables:', list(_ds.data_vars)[:6])
    if _lat_arr is not None:
        print('  Lat range : {:.4f} to {:.4f}'.format(float(_lat_arr.min()), float(_lat_arr.max())))
    if _lon_arr is not None:
        print('  Lon range : {:.4f} to {:.4f}'.format(float(_lon_arr.min()), float(_lon_arr.max())))
    _ds.close()
    print()

# US states shapefile
if _states_fp.exists():
    print('US States GeoJSON:')
    print('  CRS      :', _states_gdf.crs)
    print('  N rows   :', len(_states_gdf))
    _sb = _states_gdf.total_bounds
    print('  Bounds   : lon {:.2f}–{:.2f}, lat {:.2f}–{:.2f}'.format(_sb[0], _sb[2], _sb[1], _sb[3]))
print()
print('=> All datasets should be EPSG:4326. Pixel centres align at 0.125° steps.')

(b) Dataset grid/CRS summary

NLDAS CONUS analysis grid:
  Shape : 189 lat x 325 lon
  Res   : 0.125 deg
  CRS   : EPSG:4326 (geographic, WGS84)
  Extent: lon -124.6875–-84.1875, lat 25.8125–49.3125

OpenET GeoTIFF (OpenET_CONUS_201501.tif):
  Shape    : 189 x 325
  CRS      : EPSG:4326
  Transform: | 0.12, 0.00,-124.75|
| 0.00,-0.12, 49.38|
| 0.00, 0.00, 1.00|
  NoData   : nan

NLDAS Noah NetCDF (NLDAS_Evap_201501.nc):
  Variables: ['Evap']
  Lat range : 25.8125 to 49.3125
  Lon range : -124.6875 to -84.1875

US States GeoJSON:
  CRS      : EPSG:4326
  N rows   : 48
  Bounds   : lon -124.70–-67.00, lat 24.50–49.00

=> All datasets should be EPSG:4326. Pixel centres align at 0.125° steps.


In [11]:
# ── Diagnostic (c): Spot-check values in suspected border-artifact regions ─
# Focus on southern AZ/NM/TX (~lat 31–33 N) and northern WA/ID (~lat 48–49 N)
print('(c) Spot-check HumanET values in border-artifact regions')
print()

_region_checks = [
    ('Southern AZ/NM/TX',     (31.0, 33.0), (-115.0, -103.0)),
    ('Northern WA/ID border', (48.0, 49.5), (-124.0, (-117.0))),
]

_all_delta = np.nanmean(np.stack(list(delta_maps.values()), axis=0), axis=0)

for label, (lat_lo, lat_hi), (lon_lo, lon_hi) in _region_checks:
    _row_lo = int(np.argmin(np.abs(CONUS_LAT - lat_hi)))   # hi lat = low row index (N→S)
    _row_hi = int(np.argmin(np.abs(CONUS_LAT - lat_lo)))
    _col_lo = int(np.argmin(np.abs(CONUS_LON - lon_lo)))
    _col_hi = int(np.argmin(np.abs(CONUS_LON - lon_hi)))

    _patch = _all_delta[_row_lo:_row_hi+1, _col_lo:_col_hi+1]
    _mask_patch = crop_mask_static[_row_lo:_row_hi+1, _col_lo:_col_hi+1]
    _nonnan = _patch[np.isfinite(_patch)]
    _masked_nonnan = _patch[_mask_patch & np.isfinite(_patch)]

    print('{} (lat {:.0f}–{:.0f}N, lon {:.0f}–{:.0f}W):'.format(
        label, lat_lo, lat_hi, -lon_hi, -lon_lo))
    print('  Patch shape        : {}×{}'.format(_patch.shape[0], _patch.shape[1]))
    print('  All non-NaN vals   : {}  range: {:.1f} to {:.1f} mm/mo'.format(
        len(_nonnan), float(_nonnan.min()) if len(_nonnan) else np.nan,
        float(_nonnan.max()) if len(_nonnan) else np.nan))
    print('  CDL-masked non-NaN : {}  range: {:.1f} to {:.1f} mm/mo'.format(
        len(_masked_nonnan),
        float(_masked_nonnan.min()) if len(_masked_nonnan) else np.nan,
        float(_masked_nonnan.max()) if len(_masked_nonnan) else np.nan))
    print()

(c) Spot-check HumanET values in border-artifact regions



Southern AZ/NM/TX (lat 31–33N, lon 103–115W):
  Patch shape        : 17×97
  All non-NaN vals   : 72  range: -5.6 to 125.1 mm/mo
  CDL-masked non-NaN : 72  range: -5.6 to 125.1 mm/mo

Northern WA/ID border (lat 48–50N, lon 117–124W):
  Patch shape        : 11×57
  All non-NaN vals   : 49  range: -14.4 to 75.6 mm/mo
  CDL-masked non-NaN : 49  range: -14.4 to 75.6 mm/mo



/tmp/ipykernel_2975427/827257798.py:11: RuntimeWarning: Mean of empty slice
  _all_delta = np.nanmean(np.stack(list(delta_maps.values()), axis=0), axis=0)


### (d) Diagnosis Summary

**Root causes of the three observed problems:**

1. **Border artifacts** — The NLDAS 0.125° grid extends slightly beyond the US political boundary on all sides. The CDL cropland fraction mask was computed from NLDAS-aligned CDL rasters, which inherit these border pixels. Any NLDAS pixel whose center falls within the domain extent (lon −124.69° to −84.06°, lat 25.69° to 49.31°) is retained if ≥50% of the CDL-labeled area is cropland — even if that pixel straddles the Mexican or Canadian border. These border pixels carry real HumanET values, not fill/NoData, so they are rendered as legitimate data. **Fix: apply a hard clip of the masked raster to the dissolved 48-state US boundary after the CDL mask, so any NLDAS pixel whose centre falls outside the US polygon is zeroed to NaN.**

2. **Squished aspect ratio** — All earlier maps used `imshow(..., aspect='auto')` in EPSG:4326 geographic coordinates, which scales 1° longitude = 1° latitude regardless of true distance. At 40°N, 1° longitude ≈ 85 km while 1° latitude ≈ 111 km, so the map is compressed east–west relative to its true shape. **Fix: reproject pixel centres to EPSG:5070 (Albers Equal Area Conic, NAD83) and use `pcolormesh` with equal-aspect axes.**

3. **No state boundaries** — The earlier draft sections used bare `imshow` with no overlay. **Fix: draw state outlines from `us_states_simple.geojson` reprojected to EPSG:5070 as thin `ax.plot` lines.**

All three fixes are applied in Section 4 below.

---

## 4. Publication Figure 1 — CONUS Human ET Maps (Albers Equal Area Conic)

This section produces the two-panel publication figure using **EPSG:5070 Albers Equal Area Conic** projection, fixing three problems visible in the earlier draft maps:

1. **Border artifacts** — the equirectangular display distorts pixels near the CONUS boundary; Albers removes this distortion.
2. **Squished aspect ratio** — 1° lon ≠ 1° lat at mid-latitudes; Albers Equal Area renders CONUS with correct shape and proportions.
3. **No state boundaries** — thin gray state outlines added via a simplified CONUS states GeoJSON.

**Figure 1 panels:**
- **Panel a**: 10-year mean growing-season HumanET (April–September, 2015–2024)
- **Panel b**: Linear trend in growing-season HumanET (mm/month/year, 2015–2024)

**Requirements**: Relies on `delta_maps` (cell 1.3), `crop_mask_static` (cell 1.2), `trend_slope` / `yearly_gs` (cell 2.1) all loaded above.

### 4.1 Albers Projection Setup and State Boundaries

In [12]:
import geopandas as gpd
from pyproj import Transformer

# ── EPSG:4326 → EPSG:5070 (Albers Equal Area Conic, NAD83) ───────────────
_t5070 = Transformer.from_crs('EPSG:4326', 'EPSG:5070', always_xy=True)

# ── Projected centre coordinates for pcolormesh (shading='nearest') ───────
_LON_GRID, _LAT_GRID = np.meshgrid(CONUS_LON, CONUS_LAT)   # (189, 325)
_X5070, _Y5070 = _t5070.transform(_LON_GRID, _LAT_GRID)    # (189, 325) metres

# Display bounding box in Albers metres
_xmin, _xmax = float(_X5070.min()) - 80_000, float(_X5070.max()) + 80_000
_ymin, _ymax = float(_Y5070.min()) - 80_000, float(_Y5070.max()) + 80_000

print('Albers X range: {:.0f} to {:.0f} m'.format(_X5070.min(), _X5070.max()))
print('Albers Y range: {:.0f} to {:.0f} m'.format(_Y5070.min(), _Y5070.max()))

# ── Natural Earth boundaries (authoritative source, not hand-crafted) ─────
_NE_LAND_URL      = ('https://github.com/nvkelso/natural-earth-vector/raw/master/'
                     'geojson/ne_110m_land.geojson')
_NE_STATES_URL    = ('https://github.com/nvkelso/natural-earth-vector/raw/master/'
                     'geojson/ne_110m_admin_1_states_provinces.geojson')
_NE_COUNTRIES_URL = ('https://github.com/nvkelso/natural-earth-vector/raw/master/'
                     'geojson/ne_110m_admin_0_countries.geojson')

print('Loading Natural Earth land polygons...')
_ne_land      = gpd.read_file(_NE_LAND_URL)
print('Loading Natural Earth state boundaries...')
_ne_states    = gpd.read_file(_NE_STATES_URL)
print('Loading Natural Earth country boundaries...')
_ne_countries = gpd.read_file(_NE_COUNTRIES_URL)

# CONUS lower 48 only
_us_states_gdf = _ne_states[
    (_ne_states['admin'] == 'United States of America') &
    (~_ne_states['name'].isin(['Alaska', 'Hawaii']))
].copy()

# US + immediate neighbors for border context
_border_countries = _ne_countries[
    _ne_countries['NAME'].isin(['United States of America', 'Canada', 'Mexico'])
].copy()

print('  {:d} CONUS state polygons loaded'.format(len(_us_states_gdf)))

# ── Reproject everything to EPSG:5070 ────────────────────────────────────
_land_5070      = _ne_land.to_crs('EPSG:5070')
_states_5070    = _us_states_gdf.to_crs('EPSG:5070')
_countries_5070 = _border_countries.to_crs('EPSG:5070')

# ── Dissolve US states → CONUS boundary polygon for pixel hard-clip ───────
_us_dissolved = _us_states_gdf.dissolve()
_us_poly      = _us_dissolved.geometry.iloc[0]

# ── Build pixel-level inside US boolean mask (hard clip) ─────────────────
from shapely.geometry import Point
_boundary_mask = np.zeros((n_lat, n_lon), dtype=bool)
_cr, _cc = np.where(crop_mask_static)
for ri, ci in zip(_cr, _cc):
    pt = Point(float(CONUS_LON[ci]), float(CONUS_LAT[ri]))
    if _us_poly.contains(pt):
        _boundary_mask[ri, ci] = True

n_inside  = int(_boundary_mask.sum())
n_outside = int(crop_mask_static.sum()) - n_inside
print('Hard-clip mask: {:,} cropland pixels inside US, {:,} clipped out'.format(
    n_inside, n_outside))


def _add_basemap(ax, us_color='#DCDCDC'):
    """Basemap: white background, light gray fill inside US states only, no labels.
    Must be called after _albers_axes() so axis limits are already set.
    """
    ax.set_facecolor('white')
    _states_5070.plot(ax=ax, color=us_color, edgecolor='none', zorder=1)
    # Re-enforce limits in case geopandas adjusted them
    ax.set_xlim(_xmin, _xmax)
    ax.set_ylim(_ymin, _ymax)


def _draw_states(ax, lw=0.4, color='#333333', zorder=5):
    """Overlay Natural Earth state and country boundaries on Albers axes."""
    _states_5070.boundary.plot(ax=ax, color=color, linewidth=lw, zorder=zorder)
    _countries_5070.boundary.plot(ax=ax, color='#111111', linewidth=lw * 2,
                                  zorder=zorder + 1)


def _albers_axes(ax):
    """Remove ticks, spines, labels from a projected axes."""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect('equal')
    ax.set_xlim(_xmin, _xmax)
    ax.set_ylim(_ymin, _ymax)


Albers X range: -2860599 to 1192897 m
Albers Y range: 307156 to 3239334 m
Loading Natural Earth land polygons...


Loading Natural Earth state boundaries...


Loading Natural Earth country boundaries...


  49 CONUS state polygons loaded


Hard-clip mask: 8,919 cropland pixels inside US, 16,234 clipped out


### 4.2 Compute 10-Year Mean Growing-Season HumanET

Average all 60 growing-season monthly HumanET maps (April–September, 2015–2024). Pixels with fewer than 30 valid months (50% of 60) are masked to NaN to avoid biasing the mean toward years with sparser sampling.

In [13]:
# ── Stack all 60 growing-season HumanET maps ─────────────────────────────
_all_keys = [(y, m) for y in YEARS for m in GROWING_SEASON if (y, m) in delta_maps]
_stack    = np.stack([delta_maps[k] for k in _all_keys], axis=0)   # (60, 189, 325)

# ── Mean and valid-count ───────────────────────────────────────────────────
_n_valid      = np.sum(np.isfinite(_stack), axis=0)                 # (189, 325)
_human_et_mean = np.nanmean(_stack, axis=0)                         # (189, 325)

# Require at least 30 valid months (50%) for a reliable mean
MIN_MONTHS = 30
_human_et_mean[_n_valid < MIN_MONTHS] = np.nan

# Restrict to cropland pixels only
_human_et_mean[~crop_mask_static] = np.nan

# ── Summary statistics ─────────────────────────────────────────────────────
_valid_mean = _human_et_mean[np.isfinite(_human_et_mean)]
print('Mean HumanET summary (cropland pixels only):')
print('  N valid pixels : {:,}'.format(len(_valid_mean)))
print('  Mean           : {:.1f} mm/month'.format(float(_valid_mean.mean())))
print('  5th percentile : {:.1f} mm/month'.format(float(np.percentile(_valid_mean, 5))))
print('  95th percentile: {:.1f} mm/month'.format(float(np.percentile(_valid_mean, 95))))
print('  Max            : {:.1f} mm/month'.format(float(_valid_mean.max())))

# Color scale: 0 to 95th percentile
_mean_vmax = float(np.percentile(_valid_mean[_valid_mean > 0], 95)) if (_valid_mean > 0).any() else 50.0
print('  Color scale vmax (95th pct): {:.1f} mm/month'.format(_mean_vmax))

Mean HumanET summary (cropland pixels only):
  N valid pixels : 7,578
  Mean           : 22.4 mm/month
  5th percentile : 1.5 mm/month
  95th percentile: 59.3 mm/month
  Max            : 142.8 mm/month
  Color scale vmax (95th pct): 60.2 mm/month


/tmp/ipykernel_2975427/4102909592.py:7: RuntimeWarning: Mean of empty slice
  _human_et_mean = np.nanmean(_stack, axis=0)                         # (189, 325)


### 4.3 Publication Figure 1: Two-Panel Albers Maps

Combined publication figure saved to `figures/conus/` as both **PNG** (300 DPI) and **PDF** (vector).

Colormap choices:
- Panel a (mean HumanET): `YlOrBr` — warm sequential, suitable for positive-only irrigation intensity
- Panel b (trend): `RdBu` — diverging, blue = increasing HumanET, red = decreasing

In [14]:
import matplotlib.ticker as mticker

# ── Select the composite mask: CDL + hard US boundary clip ────────────────
# If boundary_mask is available use it; fall back to crop_mask_static only.
_final_mask = _boundary_mask if _boundary_mask is not None else crop_mask_static

# ── Trend map color scale: symmetric ±97.5th pct of |slope|, centered at 0 ─
_finite_slopes = trend_slope[_final_mask & np.isfinite(trend_slope)]
_trend_vmax    = max(float(np.nanpercentile(np.abs(_finite_slopes), 97.5)), 0.10)

# Apply final mask to both rasters
_trend_plot = np.where(_final_mask, trend_slope, np.nan)
_mean_plot  = np.where(_final_mask, _human_et_mean, np.nan)
_mean_vmin  = 0.0

# ── Colormaps (NaN = transparent so basemap shows through) ────────────────
_cmap_mean = plt.cm.YlOrBr.copy()
_cmap_mean.set_bad('white', alpha=0)

_cmap_trend = plt.cm.RdBu_r.copy()   # RdBu_r: blue=increasing, red=decreasing
_cmap_trend.set_bad('white', alpha=0)

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11})

# ── Figure layout: (14, 5) two-panel side by side ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

_PANEL_LABELS = ['a', 'b']
_PANEL_TITLES = [
    '(a) 10-year mean growing-season Human ET (2015\u20132024)',
    '(b) Trend in growing-season Human ET (2015\u20132024)',
]
_CB_LABELS = [
    'Mean Human ET [mm month\u207b\u00b9]',
    'Trend [mm month\u207b\u00b9 yr\u207b\u00b9]',
]

for idx, (ax, data, cmap, vmin, vmax) in enumerate(zip(
    axes,
    [_mean_plot,    _trend_plot],
    [_cmap_mean,    _cmap_trend],
    [_mean_vmin,    -_trend_vmax],
    [_mean_vmax,     _trend_vmax],
)):
    im = ax.pcolormesh(
        _X5070, _Y5070, data,
        cmap=cmap, vmin=vmin, vmax=vmax,
        shading='nearest', rasterized=True, zorder=2,
    )

    _albers_axes(ax)          # set limits first so basemap tiles the right extent
    _add_basemap(ax)          # ESRI NatGeo World Map at zorder=0, behind data
    _draw_states(ax, lw=0.4, color='#333333', zorder=5)   # Natural Earth boundaries

    # Horizontal colorbar below each panel
    cb = fig.colorbar(im, ax=ax, orientation='horizontal',
                      fraction=0.04, pad=0.03, shrink=0.7)
    cb.set_label(_CB_LABELS[idx], fontsize=10)
    cb.ax.tick_params(labelsize=8)

    if idx == 1:   # diverging: mark zero
        cb.set_ticks([-_trend_vmax, -_trend_vmax / 2, 0, _trend_vmax / 2, _trend_vmax])
        cb.set_ticklabels([
            '{:.1f}'.format(-_trend_vmax),
            '{:.1f}'.format(-_trend_vmax / 2),
            '0',
            '{:.1f}'.format(_trend_vmax / 2),
            '{:.1f}'.format(_trend_vmax),
        ])

    ax.text(0.02, 0.96, '(' + _PANEL_LABELS[idx] + ')',
            transform=ax.transAxes, fontsize=13, fontweight='bold',
            va='top', ha='left')
    ax.set_title(_PANEL_TITLES[idx], fontsize=11, pad=4)

fig.suptitle(
    'Human ET (OpenET \u2212 NLDAS Noah ET) across CONUS Cropland',
    fontsize=13, y=1.01, fontweight='bold',
)

plt.tight_layout(w_pad=1.0)

# ── Export ────────────────────────────────────────────────────────────────
_fig01_stem = figs / 'fig01_human_et_mean_trend'
plt.savefig(str(_fig01_stem) + '.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.savefig(str(_fig01_stem) + '.pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print('Saved:')
print('  ' + str(_fig01_stem) + '.png  (300 DPI)')
print('  ' + str(_fig01_stem) + '.pdf  (vector)')
print('Hard-clip mask applied: {}'.format(_boundary_mask is not None))


Saved:
  /home/pielab-sandbox-jcoldiron/SIF-Analysis/figures/conus/fig01_human_et_mean_trend.png  (300 DPI)
  /home/pielab-sandbox-jcoldiron/SIF-Analysis/figures/conus/fig01_human_et_mean_trend.pdf  (vector)
Hard-clip mask applied: True


---

## 5. Figure A — Per-Pixel Irrigation Buffering: Map + Scatter (Figure 2 Candidate)

This figure addresses the question: *Where* does irrigation buffer SIF under drought, and do heavily irrigated pixels show stronger buffering?

**Panel a (map):** Per-pixel OLS slope of SIF z-score ~ HumanET estimated using drought months only (SPEI ≤ −0.5, p < 0.10). Blue = positive buffering effect; red = no benefit. EPSG:5070 Albers projection with state outlines.

**Panel b (scatter):** x-axis = 10-year mean HumanET at each significant pixel (proxy for irrigation intensity); y-axis = pixel-level slope. Color = latitude (proxy for climate regime). LOWESS smoothing line shows the overall trend: do more heavily irrigated regions have stronger irrigation buffering?

### 5.1 Build Per-Pixel Data for Scatter

In [15]:
# ── Per-pixel mean HumanET and corn fraction ──────────────────────────────
# Fix: pandas groupby with multiple keys returns (key_tuple, group), not 3-tuple.
_pix_mean_et   = {}
_pix_corn_frac = {}

for (px_lat, px_lon), grp in df.groupby(['lat', 'lon']):
    _vals = grp['delta_et'].dropna()
    if len(_vals) >= 3:
        _pix_mean_et[(px_lat, px_lon)] = float(_vals.mean())
    if 'corn_frac' in grp.columns:
        _cf = grp['corn_frac'].dropna()
        if len(_cf) >= 1:
            _pix_corn_frac[(px_lat, px_lon)] = float(_cf.mean())

# ── Build scatter DataFrame from pix_sig (significant drought-month slopes) ─
_scatter_rows = []
for _, row in pix_sig.iterrows():
    _key = (round(float(row['lat']), 4), round(float(row['lon']), 4))
    _mean_et   = _pix_mean_et.get(_key, np.nan)
    _corn_frac = _pix_corn_frac.get(_key, np.nan)
    _scatter_rows.append({
        'lat':       float(row['lat']),
        'lon':       float(row['lon']),
        'slope':     float(row['slope']),
        'pval':      float(row['pval']),
        'n_obs':     int(row['n']),
        'mean_et':   _mean_et,
        'corn_frac': _corn_frac,
    })

df_scatter = pd.DataFrame(_scatter_rows)
df_scatter = df_scatter.dropna(subset=['mean_et', 'slope'])
df_scatter = df_scatter[(df_scatter['mean_et'] > -20) & (df_scatter['mean_et'] < 250)]

print('Scatter dataset: {:,} significant pixels with valid mean HumanET'.format(len(df_scatter)))
print('  Positive slope (buffering): {:,} ({:.0f}%)'.format(
    (df_scatter['slope'] > 0).sum(),
    100 * (df_scatter['slope'] > 0).mean(),
))
print('  Negative slope:             {:,} ({:.0f}%)'.format(
    (df_scatter['slope'] < 0).sum(),
    100 * (df_scatter['slope'] < 0).mean(),
))
print('  Mean HumanET range: {:.1f} to {:.1f} mm/month'.format(
    df_scatter['mean_et'].min(), df_scatter['mean_et'].max()))
print('  corn_frac available: {} pixels'.format(df_scatter['corn_frac'].notna().sum()))

Scatter dataset: 2,032 significant pixels with valid mean HumanET
  Positive slope (buffering): 1,701 (84%)
  Negative slope:             331 (16%)
  Mean HumanET range: -13.4 to 127.4 mm/month
  corn_frac available: 0 pixels


### 5.2 Figure A: Slope Map + Scatter

In [16]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.patches import Patch

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11})

# ── Build continuous slope map (signed slope, only significant pixels) ────
_slope_map_cont = np.full((n_lat, n_lon), np.nan)
_lat_to_row = {round(float(v), 4): i for i, v in enumerate(CONUS_LAT)}
_lon_to_col = {round(float(v), 4): i for i, v in enumerate(CONUS_LON)}

for _, row in pix_sig.iterrows():
    ri = _lat_to_row.get(round(float(row['lat']), 4))
    ci = _lon_to_col.get(round(float(row['lon']), 4))
    if ri is not None and ci is not None:
        _slope_map_cont[ri, ci] = row['slope']

# Color range: symmetric ±97.5th pct of |slope|
_sv = _slope_map_cont[np.isfinite(_slope_map_cont)]
_sv_max = max(float(np.percentile(np.abs(_sv), 97.5)), 0.005)

_cmap_slope = plt.cm.RdBu_r.copy()
_cmap_slope.set_bad('none')   # transparent — basemap shows through non-sig cropland

# ── Figure 2 layout ──────────────────────────────────────────────────────
_figw = 14.0
fig2 = plt.figure(figsize=(_figw, 5.5))
fig2.patch.set_facecolor('white')

# Left panel: map (60% of width)
_ax_map = fig2.add_axes([0.02, 0.08, 0.52, 0.82])
# Right cluster: main scatter + marginals
_ax_sc  = fig2.add_axes([0.62, 0.12, 0.30, 0.68])
_ax_top = fig2.add_axes([0.62, 0.81, 0.30, 0.12])   # top marginal (x)
_ax_rgt = fig2.add_axes([0.93, 0.12, 0.06, 0.68])   # right marginal (y)

# ─────────────────────────────────────────────────────────
# Panel (a): Albers slope map
# ─────────────────────────────────────────────────────────
ax = _ax_map

_use_final_mask = _boundary_mask if _boundary_mask is not None else crop_mask_static
_slope_plot2 = np.where(_use_final_mask, _slope_map_cont, np.nan)

im_a = ax.pcolormesh(
    _X5070, _Y5070, _slope_plot2,
    cmap=_cmap_slope, vmin=-_sv_max, vmax=_sv_max,
    shading='nearest', rasterized=True, zorder=2,
)

_albers_axes(ax)
_add_basemap(ax)
_draw_states(ax, lw=0.4, color='#333333', zorder=6)

cb_a = fig2.colorbar(im_a, ax=ax, orientation='horizontal',
                     fraction=0.04, pad=0.03, shrink=0.70)
cb_a.set_label('Slope: SIF z-score per mm month\u207b\u00b9 HumanET', fontsize=9)
cb_a.ax.tick_params(labelsize=7)
cb_a.set_ticks([-_sv_max, 0, _sv_max])
cb_a.set_ticklabels(['{:.3f}'.format(-_sv_max), '0', '{:.3f}'.format(_sv_max)])

# ── Map legend: gray = not significant ───────────────────────────────────
_map_legend = [
    Patch(facecolor='#DCDCDC', edgecolor='none',
          label='Not significant (p \u2265 0.10 or < {:d} drought obs)'.format(MIN_OBS_PIXEL)),
]
ax.legend(handles=_map_legend, loc='lower left', fontsize=7,
          framealpha=0.85, handlelength=1.2)

ax.text(0.02, 0.96, '(a)', transform=ax.transAxes,
        fontsize=13, fontweight='bold', va='top')
ax.set_title(
    'Per-pixel slope: SIF\u2093 \u223c Human ET | drought months (SPEI \u2264 \u22120.5)  |  p < 0.10',
    fontsize=9, pad=3,
)

# ─────────────────────────────────────────────────────────
# Panel (b): Scatter — mean HumanET vs slope
# ─────────────────────────────────────────────────────────
ax = _ax_sc

# Color by corn_frac (green colormap); fall back to steelblue if unavailable
_has_cf  = df_scatter['corn_frac'].notna().sum() > 0
_cmap_cf = plt.cm.YlGn
if _has_cf:
    _cf_vals   = df_scatter['corn_frac'].fillna(0).values
    _sc_colors = _cmap_cf(_cf_vals)
else:
    _sc_colors = 'steelblue'

ax.scatter(
    df_scatter['mean_et'], df_scatter['slope'],
    c=_sc_colors if _has_cf else None,
    color=None if _has_cf else _sc_colors,
    s=8, alpha=0.5, linewidths=0, rasterized=True, zorder=2,
)

if _has_cf:
    _sm_cf = plt.cm.ScalarMappable(cmap=_cmap_cf, norm=mcolors.Normalize(0, 1))
    _sm_cf.set_array([])
    _cb_sc = fig2.colorbar(_sm_cf, ax=ax, fraction=0.06, pad=0.02, shrink=0.8)
    _cb_sc.set_label('Corn fraction', fontsize=8)
    _cb_sc.ax.tick_params(labelsize=7)

ax.axhline(0, color='#888888', linewidth=0.9, linestyle='--', zorder=1)

ax.set_xlabel('10-year mean Human ET [mm month\u207b\u00b9]', fontsize=9)
ax.set_ylabel('Pixel slope [SIF z-score per mm month\u207b\u00b9]', fontsize=9)
ax.tick_params(labelsize=8)
ax.grid(alpha=0.2)
ax.text(-0.12, 1.04, '(b)', transform=ax.transAxes,
        fontsize=13, fontweight='bold', va='bottom')
# No title on panel (b) — avoids overlap with top marginal histogram

# ── Top marginal histogram (x = mean HumanET) ────────────────────────────
_ax_top.hist(df_scatter['mean_et'], bins=35, color='#4a90d9',
             edgecolor='none', alpha=0.75)
_ax_top.set_xlim(_ax_sc.get_xlim())
_ax_top.axis('off')

# ── Right marginal histogram (y = slope) ─────────────────────────────────
_ax_rgt.hist(df_scatter['slope'], bins=35, orientation='horizontal',
             color='#4a90d9', edgecolor='none', alpha=0.75)
_ax_rgt.set_ylim(_ax_sc.get_ylim())
_ax_rgt.axis('off')

# ── Export ────────────────────────────────────────────────────────────────
_fig02_stem = figs / 'fig02_pixel_slope_map_scatter'
fig2.savefig(str(_fig02_stem) + '.png', dpi=300, bbox_inches='tight',
             facecolor='white', edgecolor='none')
fig2.savefig(str(_fig02_stem) + '.pdf', bbox_inches='tight',
             facecolor='white', edgecolor='none')
plt.show()
print('Saved: fig02_pixel_slope_map_scatter.png / .pdf  (300 DPI)')


Saved: fig02_pixel_slope_map_scatter.png / .pdf  (300 DPI)


---

## 6. Figure B — SIF Response Across Irrigation Intensity and Drought Severity (Figure 3 Candidate)

Bins all pixel-month observations into 10 equal-count percentile groups based on HumanET value (decile 1 = lowest irrigation, decile 10 = highest). For each decile, computes the SIF z-score distribution split by drought severity.

**Drought severity categories:**
- No Drought: SPEI > −0.5
- Mild: −1.0 < SPEI ≤ −0.5
- Moderate: −1.5 < SPEI ≤ −1.0
- Severe: SPEI ≤ −1.5

**Option 1 (line plot):** x = HumanET decile, y = mean SIF z-score, one line per drought category. Directly answers: "as irrigation increases, does SIF recover more under drought?"

**Option 2 (violin plot):** x = drought severity category, half-violins colored by HumanET decile, showing the full distribution shift.

Both are exported; choose the clearest for the manuscript.

### 6.1 Compute HumanET Deciles and SIF Summaries

In [17]:
# ── Working dataset: complete cases ──────────────────────────────────────
_df_fig = df.dropna(subset=['sif_z', 'spei90d', 'delta_et']).copy()
_df_fig = _df_fig[np.isfinite(_df_fig['sif_z']) & np.isfinite(_df_fig['spei90d'])
                  & np.isfinite(_df_fig['delta_et'])].copy()

# ── HumanET decile bins (10 equal-count groups) ───────────────────────────
_df_fig['et_decile'] = pd.qcut(_df_fig['delta_et'], q=10, labels=False) + 1  # 1–10

# ── Drought severity categories ───────────────────────────────────────────
_drought_bins   = [-np.inf, -1.5, -1.0, -0.5, np.inf]
_drought_labels = ['Severe\n(SPEI<=-1.5)', 'Moderate\n(-1.5<SPEI<=-1.0)',
                   'Mild\n(-1.0<SPEI<=-0.5)', 'No Drought\n(SPEI>-0.5)']
_df_fig['drought_cat'] = pd.cut(
    _df_fig['spei90d'], bins=_drought_bins, labels=_drought_labels, right=True
)

print('HumanET decile bin boundaries (mm/month):')
_edges = _df_fig['delta_et'].quantile(np.linspace(0, 1, 11)).values
for i, (lo, hi) in enumerate(zip(_edges[:-1], _edges[1:])):
    n = (_df_fig['et_decile'] == i + 1).sum()
    print('  Decile {:2d}: {:6.1f} to {:6.1f}  (n={:,})'.format(i + 1, lo, hi, n))

print()
print('Drought category counts:')
print(_df_fig['drought_cat'].value_counts().to_string())

HumanET decile bin boundaries (mm/month):
  Decile  1: -133.4 to   -4.2  (n=43,425)
  Decile  2:   -4.2 to    3.4  (n=43,424)
  Decile  3:    3.4 to    8.9  (n=43,425)
  Decile  4:    8.9 to   13.8  (n=43,424)
  Decile  5:   13.8 to   18.6  (n=43,425)
  Decile  6:   18.6 to   23.7  (n=43,424)
  Decile  7:   23.7 to   29.7  (n=43,424)
  Decile  8:   29.7 to   37.8  (n=43,425)
  Decile  9:   37.8 to   52.1  (n=43,425)
  Decile 10:   52.1 to  233.8  (n=43,424)

Drought category counts:
drought_cat
No Drought\n(SPEI>-0.5)        313479
Mild\n(-1.0<SPEI<=-0.5)         67221
Moderate\n(-1.5<SPEI<=-1.0)     38778
Severe\n(SPEI<=-1.5)            14767


### 6.2 Figure B Option 1: Line Plot (Mean SIF by Decile × Drought Category)

In [18]:
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11})

# ── Use 20 equal-count bins for a continuous HumanET x-axis ──────────────
_df_fig['et_bin'] = pd.qcut(_df_fig['delta_et'], q=20, labels=False) + 1  # 1–20

# Median HumanET per bin — used as x-axis position (mm/month)
_bin_median_et = _df_fig.groupby('et_bin', observed=True)['delta_et'].median()

# Mean SIF z-score per (bin, drought_cat)
_grp = _df_fig.groupby(['et_bin', 'drought_cat'], observed=True)
_summary = _grp['sif_z'].agg(['mean', 'sem', 'count']).reset_index()
_summary.columns = ['et_bin', 'drought_cat', 'mean_sifz', 'se_sifz', 'n']
_summary['et_mm'] = _summary['et_bin'].map(_bin_median_et)

# ── Color palette: blue for no-drought, warm colors for drought ──────────
_drought_colors = {
    'Severe\n(SPEI<=-1.5)':             '#C62828',
    'Moderate\n(-1.5<SPEI<=-1.0)':      '#EF6C00',
    'Mild\n(-1.0<SPEI<=-0.5)':          '#F9A825',
    'No Drought\n(SPEI>-0.5)':          '#1565C0',
}
_drought_order = list(reversed(_drought_labels))   # No Drought first → Severe last

fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor('white')

for cat in _drought_order:
    _sub = _summary[_summary['drought_cat'] == cat].sort_values('et_mm')
    if len(_sub) == 0:
        continue
    col = _drought_colors.get(cat, 'gray')
    _cat_label = cat.replace('\n', ' ')
    ax.plot(_sub['et_mm'], _sub['mean_sifz'],
            color=col, linewidth=2.0, marker='o', markersize=3,
            label=_cat_label, zorder=5)
    ax.fill_between(_sub['et_mm'],
                    _sub['mean_sifz'] - _sub['se_sifz'],
                    _sub['mean_sifz'] + _sub['se_sifz'],
                    color=col, alpha=0.12, zorder=2)

# Bold dark gray zero line (solid, not dotted)
ax.axhline(0, color='#333333', linewidth=2.0, linestyle='-', zorder=3)

ax.set_xlabel('Human ET [mm month\u207b\u00b9]', fontsize=10)
ax.set_ylabel('Mean SIF z-score  (0 = climatological mean)', fontsize=10)
ax.legend(title='Drought severity', fontsize=8, title_fontsize=8,
          loc='lower right', framealpha=0.9)
ax.grid(alpha=0.25)
ax.set_title(
    'SIF Anomaly by Irrigation Intensity and Drought Severity\n'
    'CONUS Cropland Growing Season (April\u2013September, 2015\u20132024)',
    fontsize=11,
)

plt.tight_layout()
_fig03a_stem = figs / 'fig03_percentile_lines'
plt.savefig(str(_fig03a_stem) + '.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.savefig(str(_fig03a_stem) + '.pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print('Saved: fig03_percentile_lines.png / .pdf  (300 DPI)')


Saved: fig03_percentile_lines.png / .pdf  (300 DPI)


### 6.3 Figure B Option 2: Violin Plot (SIF Distribution by Drought Category, Colored by ET Decile)

In [19]:
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11})

# ── Helper: compute violin stats ──────────────────────────────────────────
def _violin_stats(data, positions, widths=0.4):
    """Compute kernel density estimate for a violin at a given position."""
    from scipy.stats import gaussian_kde
    results = []
    for pos, d in zip(positions, data):
        d = np.asarray(d)
        d = d[np.isfinite(d)]
        if len(d) < 5:
            results.append(None)
            continue
        kde      = gaussian_kde(d, bw_method='silverman')
        ymin     = float(np.percentile(d, 1))
        ymax     = float(np.percentile(d, 99))
        ys       = np.linspace(ymin, ymax, 200)
        kde_vals = kde(ys)
        kde_vals = kde_vals / kde_vals.max() * widths
        results.append({'pos': pos, 'ys': ys, 'kde': kde_vals,
                        'median': float(np.median(d)), 'n': len(d)})
    return results


# ── Only drought and no-drought groups; use 5 deciles to keep readable ────
_DECILE_SELECT = [1, 3, 5, 7, 10]
_CAT_ORDER = [
    'No Drought\n(SPEI>-0.5)',
    'Mild\n(-1.0<SPEI<=-0.5)',
    'Moderate\n(-1.5<SPEI<=-1.0)',
    'Severe\n(SPEI<=-1.5)',
]
_CAT_POSITIONS = [1, 2, 3, 4]
_CAT_SHORT     = ['No Drought', 'Mild', 'Moderate', 'Severe']

# Irrigation intensity colormap: light (low) → dark warm (high)
_irr_cmap  = plt.cm.viridis
_irr_norms = [0.05, 0.28, 0.52, 0.74, 0.95]

fig, ax = plt.subplots(figsize=(9, 5.5))
fig.patch.set_facecolor('white')

_violin_w     = 0.16
_decile_offsets = np.linspace(-0.35, 0.35, len(_DECILE_SELECT))

for di, (decile, offset, inorm) in enumerate(
        zip(_DECILE_SELECT, _decile_offsets, _irr_norms)):
    col      = _irr_cmap(inorm)
    col_dark = tuple(max(0, c - 0.15) for c in col[:3]) + (1.0,)

    for ci, (cat, cpos) in enumerate(zip(_CAT_ORDER, _CAT_POSITIONS)):
        _sub = _df_fig[(_df_fig['et_decile'] == decile) &
                       (_df_fig['drought_cat'] == cat)]['sif_z'].dropna().values
        if len(_sub) < 10:
            continue

        vstat = _violin_stats([_sub], [cpos + offset], widths=_violin_w * 0.9)
        if vstat[0] is None:
            continue
        vs = vstat[0]

        ax.fill_betweenx(vs['ys'],
                         (cpos + offset) - vs['kde'],
                         (cpos + offset) + vs['kde'],
                         color=col, alpha=0.8, linewidth=0)
        ax.plot((cpos + offset) - vs['kde'], vs['ys'],
                color=col_dark, linewidth=0.4, alpha=0.7)
        ax.plot((cpos + offset) + vs['kde'], vs['ys'],
                color=col_dark, linewidth=0.4, alpha=0.7)
        ax.hlines(vs['median'], (cpos + offset) - vs['kde'].max() * 0.7,
                  (cpos + offset) + vs['kde'].max() * 0.7,
                  color='#333333', linewidth=1.0, zorder=10)

ax.axhline(0, color='#666666', linewidth=0.8, linestyle='--', alpha=0.7, zorder=1)

_legend_patches = [
    Patch(facecolor=_irr_cmap(n), label='ET decile ' + str(d) +
          (' (low)' if d == 1 else ' (high)' if d == 10 else ''))
    for d, n in zip(_DECILE_SELECT, _irr_norms)
]
ax.legend(handles=_legend_patches, title='Human ET decile',
          loc='lower left', fontsize=7, title_fontsize=7, framealpha=0.9)

ax.set_xticks(_CAT_POSITIONS)
ax.set_xticklabels(_CAT_SHORT, fontsize=9)
ax.set_xlim(0.5, 4.5)
ax.set_xlabel('Drought severity category', fontsize=10)
ax.set_ylabel('SIF z-score  (0 = climatological mean)', fontsize=10)
ax.grid(axis='y', alpha=0.25)
ax.set_title(
    'SIF Distribution by Drought Severity and Irrigation Intensity\n'
    'CONUS Cropland Growing Season (April\u2013September, 2015\u20132024)',
    fontsize=11,
)

plt.tight_layout()
_fig03b_stem = figs / 'fig03_ridge_violin'
plt.savefig(str(_fig03b_stem) + '.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.savefig(str(_fig03b_stem) + '.pdf', bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print('Saved: fig03_ridge_violin.png / .pdf  (300 DPI)')

Saved: fig03_ridge_violin.png / .pdf  (300 DPI)
